In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("AB_NYC_2019_cleaned.csv")

features = ["price_capped", "minimum_nights", "number_of_reviews",
            "reviews_per_month", "availability_365"]
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
sil_scores = []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(list(k_range), inertias, marker="o", color="#4C72B0")
axes[0].set_title("Elbow Method: Inertia vs. k")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia (WCSS)")

axes[1].plot(list(k_range), sil_scores, marker="o", color="#DD8452")
axes[1].set_title("Silhouette Score vs. k")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.savefig("clu1_k_selection.png", dpi=150)
plt.close()

best_k_by_sil = list(k_range)[int(np.argmax(sil_scores))]

with open("clustering_notes.txt", "w") as f:
    f.write("K SELECTION\n")
    for k, inertia, sil in zip(k_range, inertias, sil_scores):
        f.write(f"k={k}: inertia={inertia:.1f}, silhouette={sil:.4f}\n")
    f.write(f"\nBest k by silhouette score: {best_k_by_sil}\n")

K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)
final_sil = silhouette_score(X_scaled, df["cluster"])

with open("clustering_notes.txt", "a") as f:
    f.write(f"\nFinal model: k={K}, silhouette score={final_sil:.4f}\n")

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df["pca1"] = X_pca[:, 0]
df["pca2"] = X_pca[:, 1]
explained_var = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(7.5, 6))
palette = sns.color_palette("Set2", K)
sns.scatterplot(x="pca1", y="pca2", hue="cluster", data=df, palette=palette, s=12, alpha=0.6, ax=ax)
ax.set_title(f"Clusters Visualized via PCA (explains {explained_var.sum()*100:.1f}% of variance)")
ax.set_xlabel(f"PC1 ({explained_var[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_var[1]*100:.1f}% variance)")
ax.legend(title="Cluster")
plt.tight_layout()
plt.savefig("clu2_pca_scatter.png", dpi=150)
plt.close()

profile = df.groupby("cluster")[features].mean().round(2)
profile["count"] = df["cluster"].value_counts().sort_index()
profile["pct"] = (profile["count"] / len(df) * 100).round(1)
profile.to_csv("cluster_profiles.csv")

with open("clustering_notes.txt", "a") as f:
    f.write("\nCLUSTER PROFILES (mean values)\n")
    f.write(str(profile) + "\n")

